# Multi-River Data Engineering Pipeline — Phases 4 to 7

**Project:** Deep Learning-Based Rainfall Data Imputation and Prediction System
**Rivers:** Sg. Johor · Sg. Kedah · Sg. Klang · Sg. Kuantan · Sg. Padas · Sg. Sarawak
**Stations:** 96 stations across 6 rivers
**Updated:** August 2026

---

This notebook documents the full data engineering pipeline for the multi-river dataset, from raw JPS/DID telemetry exports to the processed feature-ready CSVs used in Phases 4–7.

| Section | What it does | Output file |
|---|---|---|
| 1 — Raw station data | Excel → daily rainfall per station | `*_daily_raw.csv` |
| 2 — ERA5 weather mapping | NetCDF → station-matched daily weather | `era5_all_rivers_mapped.csv` |
| 3 — IMERG satellite rainfall | HDF5 → monthly satellite rainfall per station | `imerg_stations_monthly.csv` |
| 4 — Climate indices | ONI + DMI → monthly lag features | `climate_indices_monthly.csv` |
| 5 — Data quality report | Coverage, NaN rates, gap summary | — |

> **Before running this notebook:** Follow `DATA_SETUP.md` to download ERA5 (CDS) and IMERG (NASA Earthdata) data into `data/era5/` and `data/imerg_monthly/`.

## Section 1 — Raw Station Data: Excel → Daily CSV

### 1.1 Imports and Configuration

In [ ]:
import os, re, warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd
import openpyxl
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

BASE      = os.path.dirname(os.getcwd()) if os.path.basename(os.getcwd()) == "notebooks" else os.getcwd()
RAW_DIR   = os.path.join(BASE, "data", "raw", "Permohonan Data YBTM - KE", "Permohonan Data YBTM - KE")
PROC_DIR  = os.path.join(BASE, "data", "processed")
os.makedirs(PROC_DIR, exist_ok=True)

# Map Excel filename → river slug (all 6 rivers)
RIVER_FILES = {
    "johor":   "RF Sg. Johor 1.5.2015 - 1.6.2026.xlsx",
    "kedah":   "RF Sg. Kedah 1.5.2015 - 1.6.2026.xlsx",
    "klang":   "RF Sg. Klang 1.5.2015 - 1.6.2026.xlsx",
    "kuantan": "RF Sg. Kuantan 1.5.2015 - 1.6.2026.xlsx",
    "padas":   "RF Sg. Padas 1.5.2015 - 1.6.2026.xlsx",
    "sarawak": "RF Sg. Sarawak 1.5.2015 - 1.6.2026.xlsx",
}

# QC threshold: days where more than this fraction of 15-min slots are missing
# are flagged as NaN (sensor outage, not genuine zero-rain day)
MISSING_THRESHOLD = 0.25   # 25% — i.e. >6 hours missing in a day → NaN

print("Configuration:")
print(f"  Raw data directory : {RAW_DIR}")
print(f"  Output directory   : {PROC_DIR}")
print(f"  Rivers to process  : {list(RIVER_FILES.keys())}")
print(f"  QC missing threshold: {MISSING_THRESHOLD*100:.0f}% of 15-min slots")

### 1.2 Excel Format

Each river's Excel file has:
- **Rows 1–4:** Metadata header — Station Name, District, Latitude, Longitude
- **Row 5:** Column type header — alternating "Raw Daily" (15-min readings) and "Raw Yearly" (daily totals)
- **Row 6+:** Datetime (15-minute intervals), rainfall readings in mm

We read only the **"Raw Daily"** columns (15-minute tip-bucket readings) and aggregate them to daily totals ourselves, so we can apply QC (flag days with too many missing slots as NaN instead of treating them as zero-rain days).


In [ ]:
def slugify(name):
    """Convert station name to a clean Python-safe column name."""
    s = name.lower().strip()
    s = re.sub(r"[^a-z0-9]+", "_", s)
    return s.strip("_")


def parse_river_excel(river_slug, filepath):
    """
    Parse a JPS/DID river Excel file.

    Returns
    -------
    df_daily : pd.DataFrame
        Daily rainfall totals. Columns: Date, {river}_{station_slug}, ...
        NaN where >25% of 15-min slots were missing for that station-day.
    meta : list of dict
        Station metadata: slug, name, lat, lon
    """
    print(f"\nLoading {os.path.basename(filepath)} ...")
    wb = openpyxl.load_workbook(filepath, read_only=True, data_only=True)
    ws = wb.active

    # ── parse header rows (rows 1-5) ──────────────────────────────────────
    header_rows = list(ws.iter_rows(min_row=1, max_row=5, values_only=True))
    names_row   = header_rows[0]   # row 1: station names
    lat_row     = header_rows[2]   # row 3: latitudes
    lon_row     = header_rows[3]   # row 4: longitudes
    type_row    = header_rows[4]   # row 5: "Raw Daily" / "Raw Yearly"

    # Identify column indices for "Raw Daily" columns
    daily_col_indices = [i for i, v in enumerate(type_row) if v == "Raw Daily"]

    # Build station metadata from header
    meta = []
    for ci in daily_col_indices:
        name = names_row[ci]
        if not name:
            continue
        meta.append({
            "river": river_slug,
            "slug":  f"{river_slug}_{slugify(name)}",
            "name":  name,
            "lat":   lat_row[ci],
            "lon":   lon_row[ci],
            "col_idx": ci,
        })

    print(f"  Stations found: {len(meta)}")
    for m in meta:
        print(f"    {m['slug']:55s}  ({m['lat']:.4f}N, {m['lon']:.4f}E)")

    # ── read 15-min data rows (row 6 onward) ──────────────────────────────
    col_indices = [0] + [m["col_idx"] for m in meta]   # 0 = datetime column
    slugs       = [m["slug"] for m in meta]

    rows = []
    for row in ws.iter_rows(min_row=6, values_only=True):
        dt = row[0]
        if dt is None or not isinstance(dt, __import__("datetime").datetime):
            continue
        vals = [row[ci] for ci in col_indices[1:]]
        rows.append([dt] + vals)

    wb.close()

    df_15min = pd.DataFrame(rows, columns=["datetime"] + slugs)
    df_15min["datetime"] = pd.to_datetime(df_15min["datetime"])
    df_15min["date"]     = df_15min["datetime"].dt.date

    # Expected 15-min slots per day
    slots_per_day = 24 * 4   # = 96

    # ── aggregate to daily totals with QC ─────────────────────────────────
    daily_rows = []
    for date, grp in df_15min.groupby("date"):
        row_out = {"Date": pd.Timestamp(date)}
        for slug in slugs:
            vals  = grp[slug]
            n_obs = vals.notna().sum()
            # Flag day as NaN if too many 15-min slots are missing
            if n_obs < slots_per_day * (1 - MISSING_THRESHOLD):
                row_out[slug] = np.nan
            else:
                row_out[slug] = vals.sum(min_count=1)
        daily_rows.append(row_out)

    df_daily = pd.DataFrame(daily_rows)
    df_daily = df_daily.sort_values("Date").reset_index(drop=True)

    print(f"  Date range  : {df_daily['Date'].min().date()} → {df_daily['Date'].max().date()}")
    print(f"  Total days  : {len(df_daily)}")
    for slug in slugs:
        nan_pct = df_daily[slug].isna().mean() * 100
        print(f"    {slug:55s}  NaN: {nan_pct:.1f}%")

    return df_daily, meta


# ── Run for all four rivers ────────────────────────────────────────────────
all_meta   = []
river_dfs  = {}

for river, fname in RIVER_FILES.items():
    fpath = os.path.join(RAW_DIR, fname)
    if not os.path.exists(fpath):
        print(f"\nWARNING: {fpath} not found — skipping.")
        continue
    df, meta = parse_river_excel(river, fpath)
    river_dfs[river] = df
    all_meta.extend(meta)

print(f"\nTotal stations parsed: {len(all_meta)}")


### 1.3 Quality Control Notes

The 15-minute to daily aggregation applies one key QC rule:

- **Missing slot threshold (25%):** If more than 25% of the 15-minute readings within a calendar day are `NaN` (sensor offline, communication failure, or data gap), the entire day's total is set to `NaN`. This prevents treating a partially-recorded day as a zero-rainfall day.
- **Kuantan KOMTUR special case:** Station KOMTUR had 114 days of clearly erroneous readings (implausibly large values, e.g. >500 mm/day in a non-storm month). These were removed during initial QC and appear as NaN in the output.
- **Genuine zeros are preserved:** If all 96 slots in a day read 0.0 mm (a dry day), the daily total is correctly recorded as 0.0.

> **Note on Padas and Sarawak:** These East Malaysian stations have significantly higher NaN rates than the Peninsular rivers — Padas averages 40–54% missing and Sarawak 53–79%. This reflects less reliable telemetry infrastructure in those catchments, not necessarily a data problem. Stations with >60% missing days should be used with caution for model training; they can still contribute useful records during well-monitored periods.

### 1.4 Save Daily CSVs

In [ ]:
for river, df in river_dfs.items():
    out_path = os.path.join(PROC_DIR, f"{river}_daily_raw.csv")
    df.to_csv(out_path, index=False)
    print(f"Saved: {out_path}  ({df.shape[0]} rows x {df.shape[1]} cols)")

# Save station metadata
import json
meta_out = os.path.join(PROC_DIR, "all_rivers_station_meta.json")
with open(meta_out, "w") as f:
    json.dump(all_meta, f, indent=2)
print(f"Saved: {meta_out}  ({len(all_meta)} stations)")


### 1.5 Missing Data Overview

In [ ]:
fig, axes = plt.subplots(2, 3, figsize=(18, 9))
fig.suptitle("Missing Data Rate by Station and Year", fontsize=13, fontweight="bold")

for ax, (river, df) in zip(axes.flat, river_dfs.items()):
    sta_cols = [c for c in df.columns if c != "Date"]
    df2 = df.copy()
    df2["year"] = pd.to_datetime(df2["Date"]).dt.year
    nan_by_year = df2.groupby("year")[sta_cols].apply(lambda x: x.isna().mean() * 100)

    im = ax.imshow(nan_by_year.T, aspect="auto", cmap="RdYlGn_r", vmin=0, vmax=100)
    ax.set_yticks(range(len(sta_cols)))
    ax.set_yticklabels([s.replace(f"{river}_", "") for s in sta_cols], fontsize=6)
    ax.set_xticks(range(len(nan_by_year)))
    ax.set_xticklabels(nan_by_year.index, rotation=45, fontsize=8)
    ax.set_title(f"Sg. {river.title()}", fontweight="bold")
    plt.colorbar(im, ax=ax, label="NaN %")

plt.tight_layout()
plt.savefig(os.path.join(BASE, "figures", "data_eng_missing_heatmap.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Heatmap saved to figures/data_eng_missing_heatmap.png")

---

## Section 2 — ERA5 Weather Data: NetCDF → Station-Mapped CSV

ERA5 is downloaded as yearly NetCDF files (one per year, 2015–2025). For each station we find the nearest ERA5 grid point and extract daily weather variables.

**Variables extracted:**
| Variable | ERA5 name | Unit | Derived? |
|---|---|---|---|
| 2m temperature | `t2m` | K → °C | No |
| 2m dewpoint | `d2m` | K → °C | No |
| Surface pressure | `sp` | Pa | No |
| 10m U-wind | `u10` | m/s | No |
| 10m V-wind | `v10` | m/s | No |
| Total column water vapour | `tcwv` | kg/m² | No |
| Total precipitation | `tp` | m → mm | Converted |
| Relative humidity | `rh` | % | Derived from t2m, d2m |
| Wind speed | `ws` | m/s | Derived from u10, v10 |

### 2.1 Imports


In [ ]:
import netCDF4 as nc
import glob

ERA5_DIR = os.path.join(BASE, "data", "era5")
era5_files = sorted(glob.glob(os.path.join(ERA5_DIR, "era5_malaysia_*.nc")))
print(f"ERA5 files found: {len(era5_files)}")
for f in era5_files:
    print(f"  {os.path.basename(f)}")


### 2.2 Load Station Metadata and Find Nearest ERA5 Grid Points

In [ ]:
with open(os.path.join(PROC_DIR, "all_rivers_station_meta.json")) as f:
    stations = json.load(f)

# Open the first ERA5 file to read the grid
with nc.Dataset(era5_files[0]) as ds:
    era5_lat = ds.variables["latitude"][:].data
    era5_lon = ds.variables["longitude"][:].data

print(f"ERA5 grid: {len(era5_lat)} lats x {len(era5_lon)} lons")
print(f"Lat range: {era5_lat.min():.2f} – {era5_lat.max():.2f}")
print(f"Lon range: {era5_lon.min():.2f} – {era5_lon.max():.2f}")

# Find nearest grid index for each station
def nearest_idx(arr, val):
    return int(np.argmin(np.abs(arr - val)))

for sta in stations:
    li = nearest_idx(era5_lat, sta["lat"])
    loi = nearest_idx(era5_lon, sta["lon"])
    sta["era5_lat_idx"] = li
    sta["era5_lon_idx"] = loi
    sta["era5_lat"]     = float(era5_lat[li])
    sta["era5_lon"]     = float(era5_lon[loi])
    dist_km = np.sqrt(((sta["lat"] - era5_lat[li]) * 111)**2 +
                      ((sta["lon"] - era5_lon[loi]) * 111 * np.cos(np.radians(sta["lat"])))**2)
    sta["era5_dist_km"] = round(float(dist_km), 1)

print(f"\nStation → ERA5 grid mapping:")
print(f"{'Station':<55} {'Sta lat':>8} {'Sta lon':>8} {'ERA5 lat':>9} {'ERA5 lon':>9} {'Dist km':>8}")
print("-" * 100)
for sta in stations:
    print(f"{sta['slug']:<55} {sta['lat']:>8.4f} {sta['lon']:>8.4f} "
          f"{sta['era5_lat']:>9.4f} {sta['era5_lon']:>9.4f} {sta['era5_dist_km']:>7.1f}")


### 2.3 Extract Daily ERA5 Values for All Stations

In [ ]:
def extract_era5_for_year(nc_file, stations):
    """Extract daily ERA5 variables for all stations from one yearly NetCDF file."""
    rows = []
    with nc.Dataset(nc_file) as ds:
        times = nc.num2date(ds.variables["time"][:], ds.variables["time"].units)
        t2m_all  = ds.variables["t2m"][:]
        d2m_all  = ds.variables["d2m"][:]
        sp_all   = ds.variables["sp"][:]
        u10_all  = ds.variables["u10"][:]
        v10_all  = ds.variables["v10"][:]
        tcwv_all = ds.variables["tcwv"][:]
        tp_all   = ds.variables["tp"][:]

        for ti, t in enumerate(times):
            date = pd.Timestamp(t.year, t.month, t.day)
            row  = {"Date": date}
            for sta in stations:
                li  = sta["era5_lat_idx"]
                loi = sta["era5_lon_idx"]
                slug = sta["slug"]

                t2m  = float(t2m_all[ti, li, loi]) - 273.15   # K → °C
                d2m  = float(d2m_all[ti, li, loi]) - 273.15
                sp   = float(sp_all[ti, li, loi])
                u10  = float(u10_all[ti, li, loi])
                v10  = float(v10_all[ti, li, loi])
                tcwv = float(tcwv_all[ti, li, loi])
                tp   = float(tp_all[ti, li, loi]) * 1000       # m → mm

                # Derived variables
                # Relative humidity from Magnus formula
                a, b = 17.625, 243.04
                gamma_t = (a * t2m) / (b + t2m)
                gamma_d = (a * d2m) / (b + d2m)
                rh      = 100 * np.exp(gamma_d - gamma_t)
                rh      = np.clip(rh, 0, 100)
                ws      = np.sqrt(u10**2 + v10**2)

                row[f"{slug}_t2m"]  = round(t2m, 3)
                row[f"{slug}_d2m"]  = round(d2m, 3)
                row[f"{slug}_sp"]   = round(sp, 1)
                row[f"{slug}_u10"]  = round(u10, 4)
                row[f"{slug}_v10"]  = round(v10, 4)
                row[f"{slug}_tcwv"] = round(tcwv, 3)
                row[f"{slug}_tp"]   = round(max(tp, 0), 4)
                row[f"{slug}_rh"]   = round(rh, 2)
                row[f"{slug}_ws"]   = round(ws, 4)

            rows.append(row)
    return rows


# Process all years
all_rows = []
for nc_file in era5_files:
    yr = os.path.basename(nc_file).split("_")[-1].replace(".nc", "")
    print(f"Processing ERA5 {yr}...", end=" ", flush=True)
    rows = extract_era5_for_year(nc_file, stations)
    all_rows.extend(rows)
    print(f"{len(rows)} days")

df_era5 = pd.DataFrame(all_rows).sort_values("Date").reset_index(drop=True)
print(f"\nERA5 DataFrame: {df_era5.shape[0]} rows x {df_era5.shape[1]} cols")
print(f"Date range: {df_era5['Date'].min().date()} → {df_era5['Date'].max().date()}")


### 2.4 Save ERA5 Mapped CSV

In [ ]:
# Save full multi-river ERA5
out_path = os.path.join(PROC_DIR, "era5_all_rivers_mapped.csv")
df_era5.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({df_era5.shape[0]} rows x {df_era5.shape[1]} cols)")

# Kuantan uses a separate file (different ERA5 grid cells — see Section 1.2 note)
kuan_slugs = [s["slug"] for s in stations if s["river"] == "kuantan"]
kuan_cols  = ["Date"] + [c for c in df_era5.columns if any(c.startswith(s) for s in kuan_slugs)]
df_kuan    = df_era5[kuan_cols].copy()
kuan_path  = os.path.join(PROC_DIR, "era5_kuantan_station_mapped.csv")
df_kuan.to_csv(kuan_path, index=False)
print(f"Saved: {kuan_path}  ({df_kuan.shape[0]} rows x {df_kuan.shape[1]} cols)")


---

## Section 3 — IMERG Satellite Rainfall: HDF5 → Monthly CSV

NASA IMERG V07B provides monthly rainfall at 0.1° (~11 km) resolution. For each station we extract:
- **Nearest pixel** (`_imerg_mm`): single nearest 0.1° grid cell, converted from mm/hr to mm/month
- **3×3 spatial mean** (`_imerg_3x3_mm`): average of the 3×3 surrounding pixels (~0.3° box), less noisy for small catchments

### 3.1 Extract IMERG per Station


In [ ]:
import h5py
import re as _re
from calendar import monthrange

IMERG_DIR = os.path.join(BASE, "data", "imerg_monthly")
_IMERG_RE = _re.compile(r"3B-MO\.MS\.MRG\.3IMERG\.(\d{4})(\d{2})\d{2}-")

hdf_files = sorted(Path(IMERG_DIR).glob("*.HDF5"))
print(f"IMERG files found: {len(hdf_files)}")
if hdf_files:
    print(f"  First: {hdf_files[0].name}")
    print(f"  Last:  {hdf_files[-1].name}")


In [ ]:
imerg_rows = []

for hdf in hdf_files:
    m = _IMERG_RE.search(hdf.name)
    if not m:
        continue
    year, mon = int(m.group(1)), int(m.group(2))
    date  = pd.Timestamp(year=year, month=mon, day=1)
    hours = monthrange(year, mon)[1] * 24   # mm/hr → mm/month

    with h5py.File(hdf, "r") as h:
        lat_arr = h["Grid/lat"][:]        # shape (1800,) -89.95..89.95 step 0.1
        lon_arr = h["Grid/lon"][:]        # shape (3600,) -179.95..179.95 step 0.1
        prec    = h["Grid/precipitation"][0]   # (3600, 1800) mm/hr; fill=-9999.9

    row = {"Date": date}
    for sta in stations:
        slug = sta["slug"]
        slat, slon = sta["lat"], sta["lon"]

        # Nearest pixel
        lon_idx = int(np.argmin(np.abs(lon_arr - slon)))
        lat_idx = int(np.argmin(np.abs(lat_arr - slat)))

        v = prec[lon_idx, lat_idx]
        mm_pt = float(v * hours) if v > -9000 else np.nan

        # 3×3 spatial mean (±1 pixel = ±0.1°)
        lo0 = max(0, lon_idx - 1); lo1 = min(3600, lon_idx + 2)
        la0 = max(0, lat_idx - 1); la1 = min(1800, lat_idx + 2)
        patch = prec[lo0:lo1, la0:la1]
        valid = patch[patch > -9000]
        mm_3x3 = float(valid.mean() * hours) if len(valid) > 0 else np.nan

        row[f"{slug}_imerg_mm"]     = round(mm_pt, 2)  if not np.isnan(mm_pt)  else np.nan
        row[f"{slug}_imerg_3x3_mm"] = round(mm_3x3, 2) if not np.isnan(mm_3x3) else np.nan

    imerg_rows.append(row)
    print(f"  {date.strftime('%Y-%m')} processed", end="\r")

df_imerg = pd.DataFrame(imerg_rows).sort_values("Date").reset_index(drop=True)
print(f"\nIMERG DataFrame: {df_imerg.shape[0]} rows x {df_imerg.shape[1]} cols")
print(f"Date range: {df_imerg['Date'].min()} → {df_imerg['Date'].max()}")

out_path = os.path.join(PROC_DIR, "imerg_stations_monthly.csv")
df_imerg.to_csv(out_path, index=False)
print(f"Saved: {out_path}")


---

## Section 4 — Climate Indices: ONI & DMI

Two large-scale ocean-atmosphere indices are used as additional features in Phase 7. They capture ENSO (El Niño/La Niña) and Indian Ocean Dipole signals that drive year-to-year rainfall variability in Malaysia — particularly for Johor (equatorial, bimodal) and Kedah (northwest monsoon).

| Index | Source | What it measures |
|---|---|---|
| **ONI** (Oceanic Niño Index) | NOAA CPC | Pacific SST anomaly; positive = El Niño (less rain Malaysia) |
| **DMI** (Dipole Mode Index) | JAMSTEC | Indian Ocean temperature difference; positive = drier Malay Peninsula |

### 4.1 Download and Parse ONI


In [ ]:
import requests

# ── ONI from NOAA CPC ─────────────────────────────────────────────────────
print("Fetching ONI from NOAA CPC...")
oni_url = "https://www.cpc.ncep.noaa.gov/data/indices/oni.ascii.txt"
resp = requests.get(oni_url, timeout=30)
resp.raise_for_status()

from io import StringIO
oni_raw = pd.read_csv(StringIO(resp.text), sep=r"\s+", skiprows=1,
                      names=["year","jan","feb","mar","apr","may","jun",
                             "jul","aug","sep","oct","nov","dec"])

# Melt to long format (one row per month)
oni_long = oni_raw.melt(id_vars="year", var_name="month_abbr", value_name="oni")
month_map = {"jan":1,"feb":2,"mar":3,"apr":4,"may":5,"jun":6,
             "jul":7,"aug":8,"sep":9,"oct":10,"nov":11,"dec":12}
oni_long["month"] = oni_long["month_abbr"].map(month_map)
oni_long["Date"]  = pd.to_datetime(oni_long[["year","month"]].assign(day=1))
oni_long = oni_long[["Date","oni"]].sort_values("Date").reset_index(drop=True)
oni_long["oni"] = pd.to_numeric(oni_long["oni"], errors="coerce")

print(f"ONI records: {len(oni_long)}")
print(oni_long.tail(6).to_string(index=False))


In [ ]:
# ── DMI from JAMSTEC ─────────────────────────────────────────────────────
print("\nFetching DMI from JAMSTEC...")
dmi_url = "https://www.jamstec.go.jp/aplinfo/sintexf/iod/Data/dmi.monthly.txt"
try:
    resp = requests.get(dmi_url, timeout=30)
    resp.raise_for_status()
    lines = [l for l in resp.text.splitlines() if not l.startswith("#") and l.strip()]
    dmi_raw = pd.read_csv(StringIO("\n".join(lines)), sep=r"\s+",
                          header=None, names=["year","month","dmi"])
    dmi_raw["Date"] = pd.to_datetime(dmi_raw[["year","month"]].assign(day=1))
    dmi_long = dmi_raw[["Date","dmi"]].sort_values("Date").reset_index(drop=True)
    print(f"DMI records: {len(dmi_long)}")
    print(dmi_long.tail(6).to_string(index=False))
except Exception as e:
    print(f"DMI download failed ({e}). Using existing climate_indices_daily.csv instead.")
    dmi_long = None


In [ ]:
# ── Merge ONI + DMI, compute lag features, save ───────────────────────────
df_ci = oni_long.copy()
if dmi_long is not None:
    df_ci = df_ci.merge(dmi_long, on="Date", how="left")
else:
    # Load from existing processed file if download failed
    df_ci_daily = pd.read_csv(os.path.join(PROC_DIR, "climate_indices_daily.csv"),
                              parse_dates=["Date"])
    df_ci = df_ci_daily.groupby(df_ci_daily["Date"].dt.to_period("M")).first().reset_index(drop=True)
    df_ci = df_ci[["Date", "oni", "dmi"]]

df_ci = df_ci.sort_values("Date").reset_index(drop=True)

# Compute lag features (shift by 3 and 6 months)
df_ci["oni_lag3m"] = df_ci["oni"].shift(3)
df_ci["oni_lag6m"] = df_ci["oni"].shift(6)
df_ci["dmi_lag3m"] = df_ci["dmi"].shift(3)
df_ci["dmi_lag6m"] = df_ci["dmi"].shift(6)

out_path = os.path.join(PROC_DIR, "climate_indices_monthly.csv")
df_ci.to_csv(out_path, index=False)
print(f"Saved: {out_path}  ({len(df_ci)} rows)")
print(df_ci[["Date","oni","dmi","oni_lag3m","dmi_lag3m"]].tail(6).to_string(index=False))


---

## Section 5 — Data Quality Summary

A final cross-check of all processed files — confirming date ranges, NaN rates, and column counts.


In [ ]:
print("=" * 75)
print("PROCESSED DATA SUMMARY")
print("=" * 75)

files_to_check = [
    ("johor_daily_raw.csv",          "Johor daily rainfall (18 stations)"),
    ("kedah_daily_raw.csv",          "Kedah daily rainfall (11 stations)"),
    ("klang_daily_raw.csv",          "Klang daily rainfall (19 stations)"),
    ("kuantan_daily_raw.csv",        "Kuantan daily rainfall (5 stations)"),
    ("padas_daily_raw.csv",          "Padas daily rainfall (3 stations, Sabah)"),
    ("sarawak_daily_raw.csv",        "Sarawak daily rainfall (40 stations)"),
    ("era5_all_rivers_mapped.csv",   "ERA5 weather (all rivers)"),
    ("era5_kuantan_station_mapped.csv", "ERA5 weather (Kuantan only)"),
    ("imerg_stations_monthly.csv",   "IMERG satellite monthly"),
    ("climate_indices_monthly.csv",  "Climate indices (ONI+DMI+lags)"),
]

for fname, desc in files_to_check:
    fpath = os.path.join(PROC_DIR, fname)
    if not os.path.exists(fpath):
        print(f"  MISSING: {fname}")
        continue
    df = pd.read_csv(fpath, parse_dates=["Date"])
    nan_pct = df.drop(columns=["Date"]).isna().mean().mean() * 100
    print(f"  {fname:<45}  {df.shape[0]:>5} rows  {df.shape[1]:>4} cols  NaN: {nan_pct:.1f}%")
    print(f"    {desc}")
    print(f"    Date range: {df['Date'].min().date()} → {df['Date'].max().date()}")

print("\nAll processed files are ready for model training.")

### 5.2 Sample Time Series — Best Station per River

In [ ]:
fig, axes = plt.subplots(6, 1, figsize=(14, 18), sharex=False)
fig.suptitle("Sample Monthly Rainfall — Representative Station per River (processed data)", fontsize=13, fontweight="bold")

samples = [
    ("johor_daily_raw.csv",   "johor_lepau",                    "Sg. Johor — Lepau"),
    ("kedah_daily_raw.csv",   "kedah_sg_temin_di_kg_jeragan",   "Sg. Kedah — Sg. Temin"),
    ("klang_daily_raw.csv",   "klang_sg_klang_di_kg_berembang", "Sg. Klang — Kg. Berembang"),
    ("kuantan_daily_raw.csv", "kuantan_sg_cherating",           "Sg. Kuantan — Sg. Cherating"),
    ("padas_daily_raw.csv",   "padas_sg_pegalan_di_keranaan",   "Sg. Padas — Sg. Pegalan (Sabah)"),
    ("sarawak_daily_raw.csv", "sarawak_barrage",                "Sg. Sarawak — Barrage (Kuching)"),
]

for ax, (fname, col, title) in zip(axes, samples):
    fpath = os.path.join(PROC_DIR, fname)
    if not os.path.exists(fpath):
        ax.set_title(f"{title} — file not found")
        continue
    df = pd.read_csv(fpath, parse_dates=["Date"])
    if col not in df.columns:
        ax.set_title(f"{title} — column not found")
        continue
    monthly = df.set_index("Date")[col].resample("MS").sum()
    ax.bar(monthly.index, monthly.values, width=25, color="#0EA5E9", alpha=0.8)
    ax.set_title(title, fontweight="bold")
    ax.set_ylabel("Rainfall (mm)")
    ax.yaxis.set_major_formatter(mticker.FuncFormatter(lambda x, _: f"{x:.0f}"))
    ax.grid(axis="y", alpha=0.3)

plt.tight_layout()
plt.savefig(os.path.join(BASE, "figures", "data_eng_sample_timeseries.png"), dpi=120, bbox_inches="tight")
plt.show()
print("Plot saved to figures/data_eng_sample_timeseries.png")

---

## Summary

| Output file | Description | Stations | Used in |
|---|---|---|---|
| `johor_daily_raw.csv` | Daily totals, QC-flagged NaN | 18 | Phase 6, 7 |
| `kedah_daily_raw.csv` | Daily totals | 11 | Phase 6, 7 |
| `klang_daily_raw.csv` | Daily totals | 19 | Phase 6, 7 |
| `kuantan_daily_raw.csv` | Daily totals, KOMTUR QC applied | 5 | Phase 4, 5, 6, 7 |
| `padas_daily_raw.csv` | Daily totals, Sabah (high NaN — see QC note) | 3 | Future |
| `sarawak_daily_raw.csv` | Daily totals, Sarawak (high NaN — see QC note) | 40 | Future |
| `era5_all_rivers_mapped.csv` | Daily ERA5 (9 vars × all stations) | 96 | Phase 6, 7 |
| `era5_kuantan_station_mapped.csv` | Daily ERA5 (9 vars × Kuantan stations) | 5 | Phase 4, 5 |
| `imerg_stations_monthly.csv` | Monthly IMERG (2 features × all stations) | 96 | Phase 7 |
| `climate_indices_monthly.csv` | Monthly ONI + DMI + 3m/6m lags | — | Phase 7 |
| `all_rivers_station_meta.json` | Station slugs, lat/lon, ERA5 grid mapping | 96 | All phases |

*Proceed to `02b_gan_cnn_tensorflow.ipynb` for GAN imputation, then `03b_prediction_phases3to7.ipynb` for all prediction phases.*